In [1]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: /shared-docker/CyberSec-Reasoner


In [2]:
import torch
if torch.cuda.is_available():
    print(f"Number of available GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties().total_memory / 1024**3:.2f} GB")
else:
    print("No GPU detected! Using CPU...")

Number of available GPUs: 1
GPU Name: 
Total GPU Memory: 191.69 GB


In [3]:
from dataclasses import dataclass
from src.inference import CyberSecReasoner

In [4]:
@dataclass
class CyberSecReasonerConfig:
    model_path: str = "./models/qwen3.5-4b-lora-grpo/"
    # generation parameters
    max_new_tokens: int = 1024
    temperature: float = 0.7
    top_p: float = 0.9
    do_sample: bool = False
    # model loading
    torch_dtype: str = "float16"
    device_map: str = "auto"
    use_cache: bool = True
    # tokenization
    padding: bool = True
    truncation: bool = True
    padding_side: str = "left"
    repetition_penalty: str = 1.05

In [5]:
generator = CyberSecReasoner(CyberSecReasonerConfig())

TYPE: <class 'str'>
VALUE: ./models/qwen3.5-4b-lora-grpo/


`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

In [6]:
system_prompt = """You are a cybersecurity reasoning expert specialized in vulnerability analysis and classification.

Your task is to analyze CVE (Common Vulnerabilities and Exposures) descriptions and map them to the most appropriate CWE (Common Weakness Enumeration).

You must follow a structured reasoning workflow:

1. Understand the vulnerability context and affected components  
2. Identify the vulnerability type (e.g., XSS, memory corruption, improper validation)  
3. Determine the root cause of the weakness  
4. Map the root cause to the most appropriate CWE category  

Guidelines:
- Focus on root cause rather than surface-level keywords  
- Use precise cybersecurity terminology  
- Ensure reasoning clearly supports the final CWE selection  
- Prefer the most specific applicable CWE when possible  

Avoid:
- Guessing without justification  
- Contradictions between reasoning and conclusion  
- Irrelevant or overly generic explanations  
- Blindly copying CWE identifiers from the input  

Output Format (STRICT):

1. First, provide detailed step-by-step reasoning inside:
   <think> ... </think>

2. Then provide a structured analytical explanation (clear, well-organized, 1–2 paragraphs) that summarizes:
   - the vulnerability type  
   - the root cause  
   - why the selected CWE is the best match  

3. The last line must contain ONLY the CWE ID (e.g., CWE-79)

Important:
- The explanation must be consistent with the reasoning  
- The CWE must be fully justified by both reasoning and explanation  
- Do not include anything after the CWE ID  
"""

In [7]:
prompt = ('Analyze the following CVE description and map it to the appropriate CWE. Provide a brief justification for your choice. Ensure the last line of your response contains only the CWE ID.'
          'CVE Description: Cross-site scripting (XSS) vulnerability in MyBulletinBoard (MyBB) allows remote attackers to inject arbitrary web script or HTML via a signature containing a JavaScript '
          'URI in the SRC attribute of an IMG element, in which the URI uses SGML numeric character references without trailing semicolons, as demonstrated by "&#106&#97&#118&#97&#115&#99&#114&#105&#112&#116".')

In [8]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt}
]

In [9]:
output = generator.generate(messages)
reasoning, description = generator.format_completions(output)
print(reasoning)
print(description)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


<think>
Okay, let's tackle this CVE analysis. The problem is a cross-site scripting (XSS) vulnerability in MyBulletinBoard (MyBB). The description mentions that remote attackers can inject arbitrary web script or HTML through a signature with a JavaScript URI in the SRC attribute of an IMG element. The specific issue is using SGML numeric character references without a trailing semicolon.

First, I need to recall what XSS vulnerabilities generally map to in terms of CWE. XSS is commonly associated with CWE-79, which is about improper neutralization of input during web page generation. But let me make sure there's not a more specific CWE here.

The attack vector here involves injecting a URI in the SRC attribute. The example given uses numeric character references (like &#106, which is a Unicode code point) without a semicolon. Maybe the application isn't properly sanitizing these references, allowing the injection of scripts.

CWE-79 typically covers cases where user input isn't neutra